In [2]:
from openai import OpenAI
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI()

Enter your OpenAI API key:  ········


In [3]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

Hello! How can I assist you today?


In [15]:
import json
import pandas as pd
from typing import Dict, List, Any
import numpy as np

def extract_gtfs(file_path):
    json_data = Dict[str, Any]
    with open(file_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    normalized_data = []
    for poi in json_data['elements']:
        # Copy basic properties like type, id, lat, lon
        item = {k: v for k, v in poi.items() if k != 'tags'}
        if 'tags' in poi and isinstance(poi['tags'], dict):
            item.update(poi['tags'])

        normalized_data.append(item)

    df = pd.DataFrame(normalized_data)
    df = df.replace('nan', np.nan)

    if not df.empty:
        priority_cols = ['id', 'name', 'lat', 'lon', 'opening_hours', 'amenity', 'cuisine', 'wheelchair', 'toilets', 'access']
        existing_cols = [col for col in priority_cols if col in df.columns]
        other_cols = [col for col in df.columns if col not in existing_cols]
        df = df[existing_cols + other_cols]

    return df

file_path = '../data_collection/raw_data/raw_osm.json'
data_df = extract_gtfs(file_path)

In [16]:
data_df['amenity'].unique()

array([nan, 'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain',
       'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium',
       'conference_centre', 'fire_station', 'cinema', 'toilets',
       'arts_centre', 'place_of_worship'], dtype=object)

## Closed model (Open-AI)

In [29]:
# User-inputted variables
START_LAT = 32.5106
START_LON = -117.0626 
NUM_POIS_TO_VISIT = 4
TIME_PER_POI = 1.5
MAX_DIST = 15

# User Preferences
USER_PREFERENCES = {
    "start_location": {"lat": START_LAT, "lon": START_LON},
    "num_poi": NUM_POIS_TO_VISIT,
    "time_per_poi": TIME_PER_POI,
    "max_travel_dist": MAX_DIST,
    "avg_travel_speed_mph": 20.0, 
    "max_travel_time_minutes": 45.0,
    "amenity_type": "restaurant|theatre|cafe", #'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain', 'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium', 'conference_centre', 'fire_station', 'cinema', 'toilets', 'arts_centre', 'place_of_worship'
    "tourism_type": "museum|gallery|viewpoint|attraction|aquarium", # 'attraction', 'gallery', 'museum', 'viewpoint', 'artwork', 'aquarium', 'zoo', 'theme_park'
    "cuisine": "italian|mexican|american",
    "required_accessibility": ["wheelchair"], #["wheelchair", "toilets:wheelchair"],
    "visited_ids": set()
}

In [58]:
import json
import math

def haversine(lat1, lon1, lat2, lon2):
    r = 3958.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return r * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def preprocess_pois(df, current_lat, current_lon, k=250, exclude_ids=None):
    df = df.dropna(subset=["lat", "lon"]).copy()
    if exclude_ids:
        df = df[~df["id"].isin(exclude_ids)]  # skip already chosen POIs
    df["dist_miles"] = df.apply(
        lambda r: haversine(current_lat, current_lon, float(r["lat"]), float(r["lon"])),
        axis=1
    )
    df = df.sort_values("dist_miles", ascending=True).head(k)
    
    compact = []
    for _, r in df.iterrows():
        f = r.get("features", {}) if isinstance(r.get("features"), dict) else {}
        compact.append({
            "id": r.get("id"),
            "n": str(r.get("name") or "")[:60],
            "lat": r.get("lat"),
            "lon": r.get("lon"),
            "t": r.get("tourism") or r.get("amenity") or "other",
            "w": 1 if f.get("wheelchair") else 0,
        })
    return compact

def build_prompt_messages(current_lat, current_lon, generated_names, k=250):
    compact_pois = preprocess_pois(
        data_df,
        current_lat,
        current_lon,
        k=k,
        exclude_ids=generated_names
    )
    compact_json = json.dumps(compact_pois, separators=(',',':'))
    visited_list = list(generated_names)
    
    return [
        {
            "role": "system",
            "content": "You are a helpful itinerary planner. Output ONLY valid JSON, nothing else."
        },
        {
            "role": "user",
            "content": (
                f"User wants {NUM_POIS_TO_VISIT} interesting POIs starting near "
                f"({current_lat}, {current_lon}). Each visit lasts {TIME_PER_POI} hours. "
                f"Choose accessible, nearby, and diverse places.\n\n"
                f"Here are candidate POIs (compact JSON): {compact_json}\n\n"
                f"Already chosen POIs: {visited_list}\n\n"
                "Return ONE new POI (not in the visited list) as a JSON object with keys: "
                "{id, name, lat, lon, poi_type, features:{wheelchair}}."
            )
        }
    ]

In [59]:
from openai import OpenAI
import json, time, random

client = OpenAI()

def run_openai(prompt_messages, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=prompt_messages,
        temperature=0.7,
        max_tokens=512,
        response_format={"type": "json_object"}  # enforce JSON
    )
    return response.choices[0].message.content.strip()

current_lat, current_lon = START_LAT, START_LON
itinerary = []
generated_names = set()

for i in range(NUM_POIS_TO_VISIT):
    retries = 0
    max_retries = 5

    while retries < max_retries:
        prompt_messages = build_prompt_messages(current_lat, current_lon, generated_names, k=433)
        response_text = run_openai(prompt_messages)
        
        try:
            response_dict = json.loads(response_text)
        except json.JSONDecodeError:
            print("⚠️ Invalid JSON, regenerating...")
            retries += 1
            continue

        name = response_dict.get("name") or response_dict.get("n")
        if not name:
            print("⚠️ Missing name field, regenerating...")
            retries += 1
            continue

        if name in generated_names:
            print(f"⚠️ '{name}' exists, regenerating ({retries+1}/{max_retries})...")
            retries += 1
            time.sleep(random.uniform(0.5, 1.5))
            continue

        generated_names.add(name)
        itinerary.append(response_dict)
        current_lat = response_dict["lat"]
        current_lon = response_dict["lon"]
        print(f"✅ Added POI: {name}")
        break

    if retries >= max_retries:
        print(f"❌ Skipping after {max_retries} failed attempts.")

✅ Added POI: BLK Box Gallery and Creative Center
✅ Added POI: The FRONT
✅ Added POI: Cafecito 1806
✅ Added POI: Las Michoacanas


In [60]:
print(itinerary)

[{'id': 8677899984, 'name': 'BLK Box Gallery and Creative Center', 'lat': 32.5487443, 'lon': -117.0367447, 'poi_type': 'gallery', 'features': {'wheelchair': True}}, {'id': 596419020, 'name': 'The FRONT', 'lat': 32.5531682, 'lon': -117.0461124, 'poi_type': 'gallery', 'features': {'wheelchair': True}}, {'id': 12963713361, 'name': 'Cafecito 1806', 'lat': 32.5527624, 'lon': -117.0458648, 'poi_type': 'cafe', 'features': {'wheelchair': True}}, {'id': 596840123, 'name': 'Las Michoacanas', 'lat': 32.5530685, 'lon': -117.0460348, 'poi_type': 'food', 'features': {'wheelchair': False}}]


In [61]:
from evaluation import get_evaluation_metrics_verbose

metrics = get_evaluation_metrics_verbose(itinerary, USER_PREFERENCES)


--------------------------------------------------------------------------------
Itinerary
--------------------------------------------------------------------------------
Total POIs: 4
User Preferences: {'start_location': {'lat': 32.5106, 'lon': -117.0626}, 'num_poi': 4, 'time_per_poi': 1.5, 'max_travel_dist': 15, 'avg_travel_speed_mph': 20.0, 'max_travel_time_minutes': 45.0, 'amenity_type': 'restaurant|theatre|cafe', 'tourism_type': 'museum|gallery|viewpoint|attraction|aquarium', 'cuisine': 'italian|mexican|american', 'required_accessibility': ['wheelchair'], 'visited_ids': set()}
Required Features: ['wheelchair']

--------------------------------------------------------------------------------
Quality of POI Metric
--------------------------------------------------------------------------------
Total Travel Distance: 3.72 miles
Total Travel Time: 11.1 minutes (Max: 45.0 min)
Travel Distance/Time Score: 0.75
POI Diversity Score: 0.75
Preference Coverage: 0.00

----------------------